# Case Study: QuickBite Food Delivery Analysis
**ICT Academy of Kerala · Python Programming · pandas · 1-hour case study · Beginner to intermediate**

**Student version.** Write your code in the empty cells. The expected output is given under each task.

## The scenario
**QuickBite** is a food delivery start-up operating in four Kerala cities. The operations manager has exported the orders of the last **3 months** (August to October 2026) and the list of partner restaurants as two CSV files. The orders file was typed from several systems, so it contains **duplicate rows, inconsistent text, missing values and impossible delivery times**.

Your job is to **clean the data, answer the manager's questions and produce a short report** using pandas.

## Files
Keep these two files **in the same folder as this notebook** (in Google Colab, upload them with the Files panel on the left).

| File | Contents |
|---|---|
| `quickbite_orders.csv` | order_id, customer_id, restaurant_id, city, order_time, items, amount (₹), payment_mode, delivery_minutes, rating, status |
| `quickbite_restaurants.csv` | restaurant_id, restaurant_name, cuisine |

## How this case study works
| Stage | Level | Tasks | Time |
|---|---|---|---|
| A · Load and inspect | Beginner | 1 – 2 | 6 min |
| B · Clean the data | Beginner | 3 – 6 | 14 min |
| C · Analyse | Intermediate | 7 – 12 | 18 min |
| D · Insights | Intermediate | 13 – 15 | 10 min |
| E · Report | Intermediate | 16 – 17 | 8 min |
| Challenge | Optional | 18 | 4 min |

**Rules**
- Use **pandas** (and NumPy where needed). Do not edit the CSV files by hand.
- **Tasks share variables**, so run the cells in order from the top.
- Your output should **match the expected output** (it may look like a formatted table; the values must match).
- Finish with **Kernel → Restart and Run All** and make sure there are no errors.
- Text columns may show type `str` or `object` depending on your pandas version. Both are fine.

Run this cell first.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)


---
## Stage A · Load and inspect

### Task 1 · Load the data
Load `quickbite_orders.csv` into `orders` and `quickbite_restaurants.csv` into `restaurants`. Print the shape of each (orders first).

**Expected output**
```
(405, 11)
(12, 3)
```

In [ ]:
orders = pd.read_csv("quickbite_orders.csv")
restaurants = pd.read_csv("quickbite_restaurants.csv")


In [ ]:
print(orders.shape)
print(restaurants.shape)

(405, 11)
(12, 3)


### Task 2 · Find the problems
Print the columns of `orders` that have missing values together with the number missing (only columns with at least one missing value), and then the number of fully duplicated rows.

**Expected output**
```
delivery_minutes    60
rating              67
dtype: int64
5
```

In [ ]:
missing_value = orders.isna().sum()
print(missing_value[missing_value > 0])
print(orders.duplicated().sum())

delivery_minutes    49
rating              67
dtype: int64
5


---
## Stage B · Clean the data

### Task 3 · Remove duplicates
Remove the duplicate rows from `orders` (keep the first of each) and reset the row numbers. Print the new shape.

**Expected output**
```
(400, 11)
```

In [ ]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape)

(400, 11)


### Task 4 · Make the text consistent
The `payment_mode` and `city` columns have spaces and mixed upper/lower case. Strip the spaces and convert both to title case. Print the number of orders for each payment mode (sorted by the payment mode name) and then for each city (sorted by the city name).

**Expected output**
```
payment_mode
Card       98
Cash       75
Upi       184
Wallet     43
Name: count, dtype: int64
city
Kochi         174
Kozhikode      73
Thrissur       40
Trivandrum    113
Name: count, dtype: int64
```

In [ ]:
orders['payment_mode'] = orders['payment_mode'].str.strip().str.title()
orders['city'] = orders['city'].str.strip().str.title()

print(orders.groupby('payment_mode').size())
print(orders.groupby('city').size())

payment_mode
Card       98
Cash       78
Upi       186
Wallet     43
dtype: int64
city
Kochi         177
Kozhikode      73
Thrissur       41
Trivandrum    114
dtype: int64


### Task 5 · Fix delivery times
A delivery can never take more than 180 minutes, so larger values are typing errors. Replace them with missing values and print how many you replaced. Then fill the missing `delivery_minutes` of **Delivered** orders with the median delivery time of delivered orders (cancelled orders stay missing). Print the median and the number of missing `delivery_minutes` left.

**Expected output**
```
4
34.5
49
```

In [ ]:
too_high = orders['delivery_minutes'] > 180
replaced_count = too_high.sum()
orders.loc[too_high, 'delivery_minutes'] = np.nan
print(replaced_count)

delv = orders['status'] == 'Delivered'
median_del = orders.loc[delv, 'delivery_minutes'].median()
print(median_del)

orders.loc[delv, 'delivery_minutes'] = orders.loc[delv, 'delivery_minutes'].fillna(median_del)
print(orders['delivery_minutes'].isna().sum())

0
34.0
49


### Task 6 · Work with dates
Convert `order_time` to a real date-time. Add three columns: `month_no` (month number), `weekday` (day name such as Monday) and `hour`. Print the `order_time`, `month_no`, `weekday` and `hour` of the first 3 rows.

**Expected output**
```
           order_time  month_no   weekday  hour
0 2026-10-22 22:00:00        10  Thursday    22
1 2026-10-05 17:00:00        10    Monday    17
2 2026-08-24 18:00:00         8    Monday    18
```

In [ ]:
orders['order_time'] = pd.to_datetime(orders['order_time'])
orders['month_no'] = orders['order_time'].dt.month
orders['weekday'] = orders['order_time'].dt.day_name()
orders['hour'] = orders['order_time'].dt.hour

print(orders[['order_time', 'month_no', 'weekday', 'hour']].head(3))

           order_time  month_no   weekday  hour
0 2026-10-22 22:00:00        10  Thursday    22
1 2026-10-05 17:00:00        10    Monday    17
2 2026-08-24 18:00:00         8    Monday    18


---
## Stage C · Analyse

### Task 7 · Add restaurant details
Create `df` by merging `orders` with `restaurants` on `restaurant_id`, keeping every order. Print the shape of `df` and the number of orders with no `restaurant_name`.

**Expected output**
```
(400, 16)
0
```

In [ ]:
orders_clean = orders.drop_duplicates().reset_index(drop=True)
df = pd.merge(orders_clean, restaurants, on='restaurant_id', how='left')
print(df.shape)
print(df['restaurant_name'].isna().sum())

(400, 16)
0


### Task 8 · Cancellations
Print the overall cancellation percentage (1 decimal). Then print the cancellation percentage of each city (1 decimal), from highest to lowest.

**Expected output**
```
12.2
city
Trivandrum    13.3
Kochi         12.6
Kozhikode     11.0
Thrissur      10.0
Name: status, dtype: float64
```

In [ ]:
overall_pct = (df['status'] == 'Cancelled').mean() * 100
print(round(overall_pct, 1))

city_pct = df.groupby('city')['status'].apply(lambda s: (s == 'Cancelled').mean() * 100).sort_values(ascending=False)
print(city_pct.round(1))

12.1
city
Trivandrum    13.2
Kochi         12.4
Kozhikode     11.0
Thrissur       9.8
Name: status, dtype: float64


### Task 9 · Revenue
From now on use only **Delivered** orders: create `delivered` for them. Print the total revenue (sum of `amount`) and the average order value (1 decimal).

**Expected output**
```
205798.0
586.3
```

In [ ]:
delivered = df[df['status'] == 'Delivered'].copy()
total_rev = delivered['amount'].sum()
avg_val = delivered['amount'].mean()
print(total_rev)
print(round(avg_val, 1))

207834.0
583.8


### Task 10 · Cuisine revenue
Print the total revenue of each cuisine for the 3 best cuisines, from highest to lowest.

**Expected output**
```
cuisine
South Indian    51789.0
Chinese         30275.0
Biryani         29365.0
Name: amount, dtype: float64
```

In [ ]:
cuisine_rev = delivered.groupby('cuisine')['amount'].sum().sort_values(ascending=False).head(3)
print(cuisine_rev)

cuisine
South Indian    52184.0
Chinese         30275.0
North Indian    29957.0
Name: amount, dtype: float64


### Task 11 · Best restaurants
Create a table with one row per restaurant and the columns `orders` (number of orders) and `revenue` (sum of amount). Print the top 5 restaurants by revenue.

**Expected output**
```
                 orders  revenue
restaurant_name                 
Biryani House        48  29365.0
Dosa Corner          50  26812.0
Malabar Kitchen      41  24977.0
Pizza Point          32  20038.0
Wok Express          28  18618.0
```

In [ ]:
rest_summary = delivered.groupby('restaurant_name').agg(
    orders=('order_id', 'count'),
    revenue=('amount', 'sum')
)
top_5_rests = rest_summary.sort_values(by='revenue', ascending=False).head(5)
print(top_5_rests)

                 orders  revenue
restaurant_name                 
Biryani House        48  29365.0
Dosa Corner          50  26812.0
Malabar Kitchen      43  25372.0
Pizza Point          32  20038.0
Wok Express          28  18618.0


### Task 12 · City summary
For delivered orders print one table with a row per city and the columns `orders`, `avg_delivery` and `avg_rating`, rounded to 1 decimal.

**Expected output**
```
            orders  avg_delivery  avg_rating
city                                        
Kochi          152          34.6         3.8
Kozhikode       65          33.9         3.9
Thrissur        36          38.2         4.2
Trivandrum      98          32.9         4.0
```

In [ ]:
city_summary = delivered.groupby('city').agg(
    orders=('order_id', 'count'),
    avg_delivery=('delivery_minutes', 'mean'),
    avg_rating=('rating', 'mean')
).round(1)
print(city_summary)

            orders  avg_delivery  avg_rating
city                                        
Kochi          155          34.6         3.8
Kozhikode       65          33.9         3.9
Thrissur        37          38.1         4.2
Trivandrum      99          32.8         4.0


---
## Stage D · Insights

### Task 13 · Busiest day
Using **all** orders in `df`, print the number of orders for each weekday, from the busiest to the quietest.

**Expected output**
```
weekday
Monday       70
Thursday     57
Sunday       57
Wednesday    57
Saturday     56
Tuesday      53
Friday       50
Name: count, dtype: int64
```

In [ ]:
weekday_counts = df.groupby('weekday').size().sort_values(ascending=False)
print(weekday_counts)

weekday
Monday       70
Thursday     59
Sunday       58
Wednesday    58
Saturday     56
Tuesday      54
Friday       50
dtype: int64


### Task 14 · Pivot table
Print a pivot table of total revenue (delivered orders) with `city` as rows and `payment_mode` as columns, rounded to whole numbers.

**Expected output**
```
payment_mode     Card     Cash      Upi   Wallet
city                                            
Kochi         20029.0  17044.0  40501.0  11017.0
Kozhikode     10023.0   5322.0  21722.0   1760.0
Thrissur       6248.0   2954.0  10907.0   1051.0
Trivandrum    11522.0   8606.0  27555.0   9537.0
```

In [ ]:
pivot = delivered.pivot_table(index='city', columns='payment_mode', values='amount', aggfunc='sum').round(0)
print(pivot)

payment_mode     Card     Cash      Upi   Wallet
city                                            
Kochi         20029.0  18122.0  41064.0  11017.0
Kozhikode     10023.0   5322.0  21722.0   1760.0
Thrissur       6248.0   2954.0  11094.0   1051.0
Trivandrum    11522.0   8814.0  27555.0   9537.0


### Task 15 · Monthly trend and order size
Print the total revenue of each month number (delivered orders). Then create `size` with `pd.cut` on `items` using bins `[0, 2, 4, 6]` and labels `Small`, `Medium`, `Large`, and print the average amount of each size (1 decimal).

**Expected output**
```
month_no
8     76536.0
9     60517.0
10    68745.0
Name: amount, dtype: float64
size
Small     279.5
Medium    572.2
Large     919.5
Name: amount, dtype: float64
```

In [ ]:
monthly_rev = delivered.groupby('month_no')['amount'].sum()
print(monthly_rev)

delivered['size'] = pd.cut(delivered['items'], bins=[0, 2, 4, 6], labels=['Small', 'Medium', 'Large'])
print(delivered.groupby('size', observed=False)['amount'].mean().round(1))

month_no
8     77385.0
9     60517.0
10    69932.0
Name: amount, dtype: float64
size
Small     279.3
Medium    572.2
Large     917.2
Name: amount, dtype: float64


---
## Stage E · Report

### Task 16 · Print the report
Print a table of the top 5 restaurants by revenue: headings `Rank` (left, 6), `Restaurant` (left, 18), `Orders` (right, 8) and `Revenue` (right, 12), a line of 44 dashes and one row per restaurant. Show revenue with a comma as thousands separator and no decimals.

**Expected output**
```
Rank  Restaurant          Orders     Revenue
--------------------------------------------
1     Biryani House           48      29,365
2     Dosa Corner             50      26,812
3     Malabar Kitchen         41      24,977
4     Pizza Point             32      20,038
5     Wok Express             28      18,618
```

In [ ]:
print(f"{'Rank':<6}{'Restaurant':<18}{'Orders':>8}{'Revenue':>12}")
print("-" * 44)
for rank, (name, row) in enumerate(top_5_rests.iterrows(), 1):
    rev_str = f"{int(row['revenue']):,}"
    print(f"{rank:<6}{name:<18}{int(row['orders']):>8}{rev_str:>12}")

Rank  Restaurant          Orders     Revenue
--------------------------------------------
1     Biryani House           48      29,365
2     Dosa Corner             50      26,812
3     Malabar Kitchen         43      25,372
4     Pizza Point             32      20,038
5     Wok Express             28      18,618


### Task 17 · Save the clean data
Save `df` as `quickbite_clean.csv` without the index column. Load the saved file again and print its shape and the first 3 `order_id` values.

**Expected output**
```
(400, 16)
['QB1349', 'QB1002', 'QB1393']
```

In [ ]:
df.to_csv('quickbite_clean.csv', index=False)
loaded_df = pd.read_csv('quickbite_clean.csv')
print(loaded_df.shape)
print(list(loaded_df['order_id'].head(3)))

(400, 16)
['QB1349', 'QB1002', 'QB1393']


---
## Stage Challenge (optional)

### Task 18 · Repeat customers
A **repeat customer** has placed **5 or more** orders (all orders in `df`). Print the number of repeat customers and what percentage of all customers they are (1 decimal).

**Expected output**
```
47
60.3
```

In [ ]:
cust_orders = df.groupby('customer_id').size()
repeat_cust = cust_orders[cust_orders >= 5]
num_repeats = len(repeat_cust)
pct_repeats = (num_repeats / len(cust_orders)) * 100
print(num_repeats)
print(round(pct_repeats, 1))

47
60.3


---
## Submission
Save the notebook as **`YourName_QuickBite_CaseStudy.ipynb`**, run **Restart and Run All**, and submit as instructed by your trainer.

**Reflect:** which task took you the longest? Which pandas functions did you have to look up? Practise those next.